
# Loan Default Risk Analysis Project

## Objective
This project analyzes borrower FICO scores and predicts:

- Probability of Default (PD)
- Expected Loss (EL)

using bucket-based risk segmentation.

---

## Key Insights Added

### Why FICO Score?
FICO score represents the creditworthiness of a borrower.

- Higher FICO → Lower risk
- Lower FICO → Higher risk

---

### Why Bucketization?
Instead of treating every FICO score individually:

- We divide scores into groups (buckets)
- Makes risk analysis easier
- Helps banks build credit policies

---

### Why Probability of Default (PD)?
PD helps financial institutions estimate:

- Which borrowers are risky
- Future loan losses
- Credit risk exposure

---

### Why Expected Loss?
Expected Loss estimates:

Expected Loss = PD × LGD × Loan Amount

Where:
- PD = Probability of Default
- LGD = Loss Given Default
- Recovery Rate = Amount bank can recover

---

## Business Understanding
Banks use this approach for:

- Loan approval
- Risk pricing
- Credit card limits
- Portfolio risk management


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# STEP 1: LOAD DATA


df = pd.read_csv(r"C:\Users\rajpu\Downloads\Task 3 and 4_Loan_Data (1).csv")

print(df.head())


# STEP 2: SELECT FICO + DEFAULT COLUMNS


fico_scores = df['fico_score']

defaults = df['default']


# STEP 3: CREATE FICO BUCKETS


# Create 10 quantile-based buckets

df['fico_bucket'] = pd.qcut(
    df['fico_score'],
    q=10,
    duplicates='drop'
)


# STEP 4: CALCULATE DEFAULT RATE PER BUCKET


bucket_summary = df.groupby('fico_bucket').agg({

    'default': ['count', 'sum', 'mean'],
    'fico_score': ['min', 'max', 'mean']

})

bucket_summary.columns = [

    'Total Borrowers',
    'Defaults',
    'PD',
    'Min FICO',
    'Max FICO',
    'Avg FICO'
]

bucket_summary = bucket_summary.reset_index()

print(bucket_summary)


# STEP 5: VISUALIZE PD VS FICO BUCKET


plt.figure(figsize=(12,6))

plt.plot(
    bucket_summary['Avg FICO'],
    bucket_summary['PD'],
    marker='o'
)

plt.title('Probability of Default vs FICO Score')

plt.xlabel('Average FICO Score')

plt.ylabel('Probability of Default')

plt.grid(True)

plt.show()


# STEP 6: CREATE PD LOOKUP FUNCTION


def predict_pd_from_fico(fico_score):
    """
    Predict probability of default using FICO buckets
    """

    for _, row in bucket_summary.iterrows():

        if (
            fico_score >= row['Min FICO']
            and fico_score <= row['Max FICO']
        ):

            return round(row['PD'], 4)

    return None


# STEP 7: EXPECTED LOSS FUNCTION


def calculate_expected_loss(
    fico_score,
    loan_amount,
    recovery_rate=0.10
):
    """
    Calculate expected loss using FICO-based PD
    """

    pd_probability = predict_pd_from_fico(fico_score)

    lgd = 1 - recovery_rate

    expected_loss = (
        pd_probability
        * lgd
        * loan_amount
    )

    return {
        "FICO Score": fico_score,
        "Probability of Default": round(pd_probability, 4),
        "Expected Loss": round(expected_loss, 2)
    }


# STEP 8: TEST EXAMPLE


result = calculate_expected_loss(

    fico_score=620,
    loan_amount=250000

)

print(result)


# Final Business Insights

## Observations

- Borrowers with low FICO scores have higher default probability.
- High FICO borrowers are generally safer.
- Expected Loss increases when:
  - Loan amount is high
  - PD is high
  - Recovery rate is low

---

## Real-World Use Cases

This model can be used in:

- Banking
- FinTech
- Credit Risk Teams
- Loan Approval Systems
- Risk Monitoring Dashboards

---

## Possible Improvements

Future enhancements may include:

- Logistic Regression Model
- Random Forest / XGBoost
- Time-series credit risk tracking
- Dynamic risk segmentation
- SHAP explainability
